# Simmetrie residue e annullamento strutturale dei correlatori — trimero anello

Mirror computazionale, per $N=3$, di `simmetrie_correlatori_dimero.tex`/notebook. Prima di costruire
il circuito con l'ancilla, verifichiamo qui — passo per passo, con `numpy` — quali combinazioni
$(i,j,\alpha,\beta)$ del correlatore
$$C_{ij}^{\alpha\beta}(t) = \langle\psi_0|\sigma_i^\alpha(t)\,\sigma_j^\beta(0)|\psi_0\rangle$$
sono strutturalmente nulle o legate da relazioni esatte, per pura simmetria dell'Hamiltoniana.
La derivazione completa (dimostrazioni per esteso) è in `simmetrie_correlatori_trimero_anello.tex`;
qui ogni affermazione è verificata numericamente, non solo enunciata.

Punto di lavoro: $J=1,\,J'=0.4,\,b=b_c=2.4,\,D=0.15$ (Opzione B DM).

## 0. Setup

In [1]:
import numpy as np
from trimer_ring_exact import trimer_hamiltonian_dm

np.set_printoptions(precision=4, suppress=True)

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)
PAULI = {"x": X, "y": Y, "z": Z}
ETA = {"x": -1, "y": -1, "z": +1}  # V sigma^alpha V^dag = eta_alpha sigma^alpha, V=Rz(pi)=-iZ

def site_op(site, alpha):
    ops = [I2, I2, I2]
    ops[site - 1] = PAULI[alpha]
    return np.kron(np.kron(ops[0], ops[1]), ops[2])

J, Jp, b, D = 1.0, 0.4, 2.4, 0.15
H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
E, Vmat = np.linalg.eigh(H)
psi0 = Vmat[:, 0]
psi0 = psi0 * np.exp(-1j * np.angle(psi0[np.argmax(np.abs(psi0))]))  # fase reale

print(f"Punto di lavoro: J={J}, J'={Jp}, b={b}, D={D} (Opzione B)")
print(f"E0={E[0]:.6f}   gap E1-E0={E[1]-E[0]:.6f}")
print(f"max|Im psi0| = {np.max(np.abs(psi0.imag)):.2e}  (atteso: 0, fase fissata reale)")

Punto di lavoro: J=1.0, J'=0.4, b=2.4, D=0.15 (Opzione B)
E0=-5.534370   gap E1-E0=0.254700
max|Im psi0| = 0.00e+00  (atteso: 0, fase fissata reale)


## 1. $S_z^{tot}$ non è conservato per $D\neq0$

Lo scambio isotropo e il campo Zeeman commutano sempre con $S_z^{tot}$; verifichiamo che sia
solo il termine DM a romperlo.

In [2]:
Sz_tot = site_op(1, "z") + site_op(2, "z") + site_op(3, "z")
H_noDM = trimer_hamiltonian_dm(J, Jp, b, None, 0.0).to_matrix()
H_DM_only = H - H_noDM

comm_noDM = Sz_tot @ H_noDM - H_noDM @ Sz_tot
comm_DM = Sz_tot @ H_DM_only - H_DM_only @ Sz_tot
print(f"||[Sz_tot, H_ex+H_Z]|| = {np.linalg.norm(comm_noDM):.2e}  (atteso: 0 esatto)")
print(f"||[Sz_tot, H_DM]||     = {np.linalg.norm(comm_DM):.2e}  (atteso: != 0, D={D})")

||[Sz_tot, H_ex+H_Z]|| = 0.00e+00  (atteso: 0 esatto)
||[Sz_tot, H_DM]||     = 1.38e+00  (atteso: != 0, D=0.15)


## 2. Il tentativo fallito: $U_\text{naive} = \mathrm{SWAP}_{12}\cdot(V\otimes V\otimes\mathbb I)$

Nel dimero la riparazione della simmetria di scambio (rotta dal DM) era $V\otimes V$ su
**entrambi** i qubit. L'estensione ingenua al trimero — ruotare solo i due siti scambiati,
lasciare il terzo intatto — non funziona: verifichiamo che non commuta con $H$.

In [3]:
def swap12():
    S = np.zeros((8, 8), dtype=complex)
    for q1 in range(2):
        for q2 in range(2):
            for q3 in range(2):
                idx_in = q1 * 4 + q2 * 2 + q3     # base |site1 site2 site3>
                idx_out = q2 * 4 + q1 * 2 + q3    # site1 <-> site2
                S[idx_out, idx_in] = 1.0
    return S

V = -1j * Z  # Rz(pi) a meno di fase globale
SWAP12 = swap12()
U_naive = SWAP12 @ np.kron(np.kron(V, V), I2)

comm_naive = U_naive @ H @ U_naive.conj().T - H
print(f"||U_naive H U_naive^dag - H|| = {np.linalg.norm(comm_naive):.2f}  (non piccolo: fallisce)")

||U_naive H U_naive^dag - H|| = 4.55  (non piccolo: fallisce)


## 3. La costruzione corretta: $U_\text{anello} = \mathrm{SWAP}_{12}\cdot V^{\otimes3}$

Applicando $V=R_z(\pi)$ anche al sito spettatore (sito 3), la simmetria torna esatta.

In [4]:
U_anello = SWAP12 @ np.kron(np.kron(V, V), V)

comm = U_anello @ H @ U_anello.conj().T - H
print(f"||[U_anello, H]|| = {np.linalg.norm(comm):.2e}  (atteso: 0 a precisione macchina)")

# ripetuto su punti casuali per non dipendere dal solo punto di lavoro
rng = np.random.default_rng(0)
max_norm = 0.0
for _ in range(20):
    Jr, Jpr, br, Dr = rng.uniform(-2, 2, 4)
    Hr = trimer_hamiltonian_dm(Jr, Jpr, br, "B", Dr).to_matrix()
    max_norm = max(max_norm, np.linalg.norm(U_anello @ Hr @ U_anello.conj().T - Hr))
print(f"max su 20 punti casuali (J,J',b,D): {max_norm:.2e}")

U2 = U_anello @ U_anello
print(f"\n||U_anello^2 - (-I)|| = {np.linalg.norm(U2 + np.eye(8)):.2e}  "
      f"(atteso: U^2=-I, non un'involuzione come nel dimero)")

||[U_anello, H]|| = 0.00e+00  (atteso: 0 a precisione macchina)
max su 20 punti casuali (J,J',b,D): 0.00e+00

||U_anello^2 - (-I)|| = 0.00e+00  (atteso: U^2=-I, non un'involuzione come nel dimero)


## 4. Lemma: azione di $U_\text{anello}$ su un operatore di singolo sito

$U_\text{anello}\,\sigma_i^\alpha\,U_\text{anello}^\dagger = \eta_\alpha\,\sigma_{\pi(i)}^\alpha$,
$\pi=(1\,2)$ (fissa il sito 3), $\eta_x=\eta_y=-1,\eta_z=+1$.

In [5]:
pi_map = {1: 2, 2: 1, 3: 3}
print(f"{'sito':>4} {'alpha':>6} {'||U s U^dag - eta*s_pi(i)||':>30}")
max_err = 0.0
for i in (1, 2, 3):
    for alpha in ("x", "y", "z"):
        lhs = U_anello @ site_op(i, alpha) @ U_anello.conj().T
        rhs = ETA[alpha] * site_op(pi_map[i], alpha)
        err = np.linalg.norm(lhs - rhs)
        max_err = max(max_err, err)
        print(f"{i:>4} {alpha:>6} {err:>30.2e}")
print(f"\nerrore massimo: {max_err:.2e}")

sito  alpha    ||U s U^dag - eta*s_pi(i)||
   1      x                       0.00e+00
   1      y                       0.00e+00
   1      z                       0.00e+00
   2      x                       0.00e+00
   2      y                       0.00e+00
   2      z                       0.00e+00
   3      x                       0.00e+00
   3      y                       0.00e+00
   3      z                       0.00e+00

errore massimo: 0.00e+00


## 5. Corollario del sito fisso: zero rigoroso per ogni $t$

Il correlatore classico esatto via formula spettrale, come riferimento per tutto il resto del
notebook e per il notebook "tutte" che segue.

In [6]:
def a_k_b_k(i, alpha, j, beta):
    A = site_op(i, alpha)
    B = site_op(j, beta)
    a = Vmat.conj().T @ (A @ psi0)
    bvec = Vmat.conj().T @ (B @ psi0)
    return np.conj(a) * bvec  # <psi0|A|k> = conj(<k|A|psi0>)

def correlatore_classico(i, alpha, j, beta, t_grid):
    prod = a_k_b_k(i, alpha, j, beta)
    phases = np.exp(1j * np.outer(t_grid, E[0] - E))
    return phases @ prod

zeri_attesi = [(3, "x", 3, "z"), (3, "z", 3, "x"), (3, "y", 3, "z"), (3, "z", 3, "y")]
print("Corollario del sito fisso -- i quattro zeri strutturali (per ogni t):\n")
for (i, alpha, j, beta) in zeri_attesi:
    pesi = a_k_b_k(i, alpha, j, beta)
    print(f"  C_{i}{j}^{alpha}{beta}: max|a_k b_k| = {np.max(np.abs(pesi)):.2e}  (atteso: 0)")

Corollario del sito fisso -- i quattro zeri strutturali (per ogni t):

  C_33^xz: max|a_k b_k| = 7.77e-17  (atteso: 0)
  C_33^zx: max|a_k b_k| = 7.77e-17  (atteso: 0)
  C_33^yz: max|a_k b_k| = 7.97e-17  (atteso: 0)
  C_33^zy: max|a_k b_k| = 7.97e-17  (atteso: 0)


## 6. Time-reversal $K$: $H$ è reale, zeri a $t=0$

$H$ reale in base computazionale $\Rightarrow$ per $i\neq j$ con esattamente una componente $y$,
$C_{ij}^{\alpha\beta}(0)=0$. Sono $\binom{3}{2}\times2\times4=24$ combinazioni.

In [7]:
print(f"max|Im H| = {np.max(np.abs(H.imag)):.2e}  (atteso: 0, H reale)\n")

count = 0
max_abs_t0 = 0.0
for i in (1, 2, 3):
    for j in (1, 2, 3):
        if i == j:
            continue
        for alpha in ("x", "y", "z"):
            for beta in ("x", "y", "z"):
                if (alpha == "y") != (beta == "y"):  # esattamente una y
                    c0 = correlatore_classico(i, alpha, j, beta, np.array([0.0]))[0]
                    count += 1
                    max_abs_t0 = max(max_abs_t0, abs(c0))
print(f"Combinazioni verificate (i!=j, esattamente una y): {count}  (atteso: 24)")
print(f"max|C(0)| su queste: {max_abs_t0:.2e}  (atteso: 0 a precisione macchina)")

max|Im H| = 0.00e+00  (atteso: 0, H reale)

Combinazioni verificate (i!=j, esattamente una y): 24  (atteso: 24)
max|C(0)| su queste: 3.89e-16  (atteso: 0 a precisione macchina)


## 7. Scan completo delle 81 combinazioni

In [8]:
sites = (1, 2, 3)
comps = ("x", "y", "z")
righe = []
for i in sites:
    for alpha in comps:
        for j in sites:
            for beta in comps:
                pesi = a_k_b_k(i, alpha, j, beta)
                righe.append({"i": i, "alpha": alpha, "j": j, "beta": beta,
                              "max|a_k b_k|": np.max(np.abs(pesi))})

import pandas as pd
df = pd.DataFrame(righe)
nulli = df[df["max|a_k b_k|"] < 1e-10]
print(f"Combinazioni totali: {len(df)}")
print(f"Nulle per ogni t (max|a_k b_k| < 1e-10): {len(nulli)}  (atteso: 4, i predetti dal Corollario)\n")
print(nulli[["i", "alpha", "j", "beta"]].to_string(index=False))

print("\nLe 5 combinazioni piu' 'ricche' (max|a_k b_k| piu' grande):")
print(df.sort_values("max|a_k b_k|", ascending=False).head(5).to_string(index=False))

Combinazioni totali: 81
Nulle per ogni t (max|a_k b_k| < 1e-10): 4  (atteso: 4, i predetti dal Corollario)

 i alpha  j beta
 3     x  3    z
 3     y  3    z
 3     z  3    x
 3     z  3    y

Le 5 combinazioni piu' 'ricche' (max|a_k b_k| piu' grande):
 i alpha  j beta  max|a_k b_k|
 3     z  3    z      0.999289
 3     z  1    x      0.676044
 1     x  3    z      0.676044
 3     z  2    x      0.676044
 2     x  3    z      0.676044


## 8. Riepilogo

- $U_\text{naive}$ (rotazione solo sui due siti scambiati) **non** è simmetria di $H$: serve
  ruotare anche il sito spettatore.
- $U_\text{anello}=\mathrm{SWAP}_{12}\cdot V^{\otimes3}$ **è** simmetria esatta, $U^2=-\mathbb I$
  (non involuzione, a differenza del dimero).
- Il sito 3 (punto fisso dello scambio) produce **4 zeri esatti per ogni $t$** — tipo di zero
  impossibile nel dimero (nessun punto fisso con solo 2 siti).
- La simmetria di time-reversal $K$ dà **24 zeri a $t=0$** (non per ogni $t$), identica nella
  struttura al dimero.
- Lo scan completo conferma: esattamente 4 nulle su 81, nessuna coincidenza numerica ulteriore.

**Prossimo passo**: `correlazioni_trimero_anello_esplorazione.ipynb` — costruzione ed esecuzione
del circuito con l'ancilla, usando queste simmetrie come base per interpretare i risultati.